## Asyncio & Non-Blocking Event Loops

Async programming is essential for high-throughput model serving, distributed RL inference servers, streaming token responses, and handling tens of thousands of concurrent I/O connections on a single OS thread.

Async programming lets one program start something that takes time, and while waiting, work on something else instead of sitting idle.

Don't think:

Async = faster CPU computation.

Think:

Async = don't waste time sitting idle while waiting for I/O.

I/O includes things like:

network requests
database queries
reading from sockets
waiting for another server
streaming data

2. The Core Building Blocks
Coroutines (async def): Functions defined with async def. Calling one returns a coroutine object (it does not execute immediately).

The await Expression: Passes control back to the event loop. Can only be used inside an async def function, and can only await:

Other coroutines.

Tasks (asyncio.Task).

Futures (asyncio.Future).

Tasks (asyncio.create_task(coro)): Wraps a coroutine into a Task object and schedules it immediately onto the event loop to run concurrently in the background.

Gathering (asyncio.gather(*tasks)): Runs multiple awaitables concurrently and aggregates their return values in order.

Python

In [ ]:
import aiohttp
import asyncio

async def hello():
    print("Hello")
    await asyncio.sleep(2)
    print("Hello")
asyncio.run(hello())
#->> in .py files

RuntimeError: asyncio.run() cannot be called from a running event loop

In [2]:
import aiohttp
import asyncio

async def hello():
    print("Hello")
    await asyncio.sleep(2)
    print("Hello")
await hello()
#->> in .py files

Hello
Hello


In [5]:
import asyncio
import time

async def model_request(request_id):
    print(f"request for {request_id} started")
    
    await asyncio.sleep(2)
    
    print(f"req for {request_id} done")
    return(f"{request_id} resource")

async def main():
    print("starting model request")
    start = time.perf_counter()
    results = await asyncio.gather(
        model_request(1),
        model_request(2),
        model_request(3)
    )
    end = time.perf_counter()
    print(results)
    print("total time",end-start)
await main()
    

starting model request
request for 1 started
request for 2 started
request for 3 started
req for 1 done
req for 2 done
req for 3 done
['1 resource', '2 resource', '3 resource']
total time 2.0065997000056086


asyncio is about concurrency, not magic parallel CPU execution

Exercise: now the important part

Don't use asyncio.gather().

Build this yourself:

In [16]:
import asyncio
import time

async def request(req_id):
    print("request for ", req_id, " started")
    
    await asyncio.sleep(2)
    
    print("request for ",req_id," done")
    return req_id

async def main():
    
    print("processing  req starts")
    
    processes = []
    results = []
    
    start = time.perf_counter()
    
    for i in range(4):
        processes.append(asyncio.create_task(request(i)))
    
    results = [await process for process in processes]
    # inbuilt->> results = await asyncio.gather(*processes)
    end = time.perf_counter()
    print(results)
    print("time taken ",end-start)

In [17]:
await main()

processing  req starts
request for  0  started
request for  1  started
request for  2  started
request for  3  started
request for  0  done
request for  2  done
request for  1  done
request for  3  done
[0, 1, 2, 3]
time taken  2.0135487000079593
